# 02 · Zonas de velocidad y área de la zona de transición (descompresión)

En esta descarga intermitente hay **tres regímenes de velocidad**:
- **ESTÁTICO** — las bolitas están quietas.
- **TRANSICIÓN / descompresión** — recién empiezan a moverse "un poquito": más rápido que
  estático pero todavía sin caer.
- **CAÍDA** — ya cayendo rápido.

**Meta (pregunta del director): medir el ÁREA de la zona de transición.**

La velocidad de transición es un **rango**, y lo sacamos de los propios datos: la distribución de
velocidades es **trimodal** (un pico estático, un pico de transición y un pico de caída). El rango
de transición va del valle estático al valle entre los dos picos móviles.

Nota: sólo seguimos las bolitas **blancas** (trazadoras) dentro del hidrogel transparente, así que
NO se puede medir densidad/empaquetamiento — trabajamos con velocidad. Calibración FIJI: 14 px = 1 cm.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from matplotlib.colors import ListedColormap

FPS  = 80
PXMM = 1.4               # 14 px = 1 cm
k    = FPS / PXMM        # px/frame -> mm/s

OUTDIR = '../outputs/nb02'          # salidas de ESTE notebook, para no ensuciar outputs/
os.makedirs(OUTDIR, exist_ok=True)

t = pd.read_pickle('../outputs/t_velocities.pkl')
print('muestras:', len(t), '| particulas:', t.particle.nunique(), f'| factor x{k:.1f} px/f->mm/s')

## 0. Control de calidad del tracking (QC)

Antes de medir velocidades, primero **miramos** el tracking crudo (0.1): una animación de las
bolitas cayendo, **sin etiquetar nada**, sólo para ver con los ojos si algo se comporta raro. Lo que
saltó a la vista ahí —bolitas que parecen "duplicarse" y ráfagas de movimiento cerca del final— es
lo que motiva las tres preguntas que respondemos con datos a continuación:

1. **¿Cómo evoluciona la descarga en el tiempo?** — ¿el silo se vacía?, ¿hasta cuándo hay
   suficientes bolitas para hacer estadística? (0.2)
2. **¿Son confiables las detecciones y el linkeo?** — ¿las bolitas "saltan" (mislinks), o el tracking
   es sano? ¿de dónde sale la "duplicación"? (0.3)
3. **Las ráfagas de movimiento del final, ¿son reales o un artefacto?** (0.4)

De ahí sale el único parámetro que fijamos en esta sección: `N_MIN`, el mínimo de bolitas por frame
para confiar en una estadística espacial.

### 0.1 Preview del tracking — mirar antes de medir

Primero lo más simple: **ver** las bolitas moverse. Cada punto es una detección; **no etiquetamos
nada** (ni estático, ni caída) — sólo queremos comprobar a ojo que el tracking sigue bolitas reales
y detectar cualquier cosa rara antes de confiar en los números.

Para eso definimos `render_particle_movie(...)`, una **función reutilizable** que renderiza un mp4
de las posiciones frame a frame (la volvemos a usar más adelante). Acepta una columna opcional para
colorear y un gancho `bg` para, en el futuro, dibujar el video real detrás de los puntos y validar
contra la realidad.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import Video

def render_particle_movie(df, frames=None, step=4, color=None, cmap='viridis', clim=None,
                          s=12, c='#222222', alpha=.8, bg=None, figsize=(5, 9),
                          out=None, fps=25, dpi=80, title=''):
    """Renderiza un mp4 de las bolitas trackeadas, frame a frame. REUTILIZABLE.

    df     : tabla con columnas 'x','y','frame' (y, opcional, la columna `color`).
    frames : frames a animar (default: 0..max submuestreado de a `step`).
    color  : nombre de columna para colorear los puntos (default: color plano `c`, SIN etiquetar).
    clim   : (min,max) para la escala de color, si se usa `color`.
    bg     : funcion frame->imagen 2D para dibujar el video real detras (gancho a futuro; default None).
    out    : ruta del mp4 (default: OUTDIR/movie.mp4). Devuelve un IPython.display.Video embebido.
    """
    out = out or f'{OUTDIR}/movie.mp4'
    if frames is None:
        frames = range(0, int(df['frame'].max()) + 1, step)
    frames = list(frames)
    by   = dict(tuple(df.groupby('frame')))
    xlim = (df.x.min() - 10, df.x.max() + 10)
    ylim = (df.y.max() + 10, df.y.min() - 10)                  # y invertido (coords de imagen)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal')
    ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
    imbg = ax.imshow(bg(frames[0]), cmap='gray', extent=[xlim[0], xlim[1], ylim[0], ylim[1]], zorder=0) if bg else None
    if color is not None:
        sc = ax.scatter([], [], s=s, cmap=cmap, alpha=alpha, edgecolors='none', zorder=2)
        if clim: sc.set_clim(*clim)
    else:
        sc = ax.scatter([], [], s=s, c=c, alpha=alpha, edgecolors='none', zorder=2)
    ttl = ax.set_title(title)

    def upd(i):
        fr = frames[i]; d = by.get(fr)
        sc.set_offsets(d[['x', 'y']].values if d is not None else np.empty((0, 2)))
        if color is not None and d is not None: sc.set_array(d[color].values)
        if bg is not None: imbg.set_data(bg(fr))
        ttl.set_text(f'{title}\nframe {fr} | t={fr/FPS:.1f}s | N={0 if d is None else len(d)}')

    FuncAnimation(fig, upd, frames=len(frames), interval=1000/fps, blit=False).save(
        out, writer='ffmpeg', fps=fps, dpi=dpi)
    plt.close(fig)
    return Video(out, embed=True)

# preview crudo de TODA la descarga, sin etiquetar (1 de cada 4 frames para que el mp4 sea liviano)
render_particle_movie(t, step=1, out=f'{OUTDIR}/tracking_preview.mp4', title='Preview del tracking (crudo, sin etiquetar)')

### 0.2 La descarga en el tiempo — el silo se vacía

Contamos cuántas bolitas trazadoras se ven en cada frame. Si el silo se descarga, ese número debe
**bajar** con el tiempo. Eje x en **frames** (video a 80 fps → `t = frame/80`, total ≈ 92 s).

Lo importante para el análisis: cuando quedan **muy pocas** bolitas, cualquier promedio espacial
(por ejemplo un área) se apoya en un puñado de puntos y deja de ser confiable. Definimos
`N_MIN = 50`: por debajo de eso quedan del orden de **~5 bolitas móviles por frame**, insuficientes
para delimitar una zona. Esos frames (~el último 10 % del video) **no se borran** — se **marcan**
como baja confianza y se excluyen sólo de las métricas cuantitativas por-frame.

In [ ]:
N_MIN = 50                                 # min bolitas/frame para confiar en una estadistica espacial
V_MOV = 51                                 # mm/s, umbral estatico/movil (se re-deriva formal en la sec. 1)

ppf   = t.groupby('frame').size()          # bolitas por frame
mov0  = t[t['v'] * k > V_MOV]
mpf   = mov0.groupby('frame').size().reindex(ppf.index, fill_value=0)   # bolitas MOVILES por frame
fmax  = int(t.frame.max())
n0    = ppf.iloc[:100].median()            # llenado inicial
low   = ppf.index[ppf < N_MIN]
f_low = int(low.min()) if len(low) else fmax
frame_ok = ppf >= N_MIN                     # <- mascara reutilizable aguas abajo (sec. 5)

print(f'llenado inicial: {n0:.0f} bolitas/frame  ->  final: {ppf.tail(1).iloc[0]:.0f}   (el silo se vacia)')
print(f'N < {N_MIN} desde el frame {f_low} (t = {f_low/FPS:.0f} s): quedan {ppf[f_low]:.0f} bolitas = {ppf[f_low]/n0*100:.0f}% del material')
print(f'frames marcados baja-confianza: {(~frame_ok).sum()} de {len(ppf)} ({(~frame_ok).mean()*100:.0f}%) | mediana de MOVILES/frame ahi: {mpf[~frame_ok].median():.0f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ppf.index, ppf.values, color='steelblue', lw=1.2, label='todas')
ax.plot(mpf.index, gaussian_filter1d(mpf.values.astype(float), 8), color='darkorange', lw=1.2, label=f'moviles (>{V_MOV} mm/s)')
ax.axhline(N_MIN, color='red', ls=':', lw=1)
ax.axvspan(f_low, fmax, color='red', alpha=.08)
ax.annotate(f'baja confianza\nN<{N_MIN}  (t>{f_low/FPS:.0f}s)', xy=(f_low, N_MIN), xytext=(f_low*0.62, n0*0.6),
            fontsize=9, color='darkred', arrowprops=dict(arrowstyle='->', color='darkred'))
ax.set_xlim(0, fmax); ax.set_xlabel('frame  (t = frame / 80 s,  total ≈ 92 s)'); ax.set_ylabel('bolitas por frame')
ax.set_title('El silo se vacia con el tiempo'); ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

### 0.3 ¿Es confiable el tracking? — dos chequeos

En el preview (0.1) parece haber bolitas "duplicadas". Antes de creerlo lo verificamos con dos
medidas:

- **¿Las bolitas saltan?** Para cada bolita miramos cuánto se desplaza entre frames consecutivos. Si
  el linkeo estuviera mal (una id que salta de una bolita a otra), veríamos saltos grandes y
  frecuentes. **Panel izquierdo (ECDF de los saltos).**
- **¿Cuántos tracks se fragmentan?** Si el tracker pierde una bolita unos frames y la recupera, la
  numera como **id nueva** → dos puntos cercanos aparecen a lo largo del tiempo (la falsa
  "duplicación"). Contamos qué fracción de tracks tiene al menos un hueco, y cuántos tracks nacen y
  mueren en cada tramo. **Panel derecho (nacimientos/muertes).**

Si (a) casi no hay saltos y (b) sí hay fragmentación y mucho recambio al final, entonces la
"duplicación" es un efecto de **fragmentación + vaciado**, no un error de detección.

In [ ]:
s = t.sort_values(['particle', 'frame'])
g = s.groupby('particle')

jump  = np.hypot(g['x'].diff(), g['y'].diff()).dropna()          # salto de una misma bolita [px/frame]
frag  = (g['frame'].diff() > 1).groupby(s['particle']).any().mean()   # % tracks con >=1 hueco
birth = g['frame'].min(); death = g['frame'].max()               # frame de nacimiento / muerte de cada track

print('SALTOS de una misma bolita (px/frame):')
print(f'  mediana {np.median(jump):.2f} | p95 {np.percentile(jump,95):.1f} | p99 {np.percentile(jump,99):.1f} | saltos >100px: {(jump>100).sum()} de {len(jump)} ({(jump>100).mean()*100:.3f}%)')
print(f'  => sin teletransportes: el linkeo es sano')
print(f'FRAGMENTACION: {frag*100:.0f}% de los tracks tienen al menos un hueco  => reapariciones = ids nuevas = falsa "duplicacion"')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].ecdf(jump.values, color='steelblue')
ax[0].axvline(1, color='gray', ls=':', lw=1)
ax[0].set_xscale('symlog', linthresh=1); ax[0].set_xlim(0, jump.max())
ax[0].set_xlabel('salto por frame [px]  (escala symlog)'); ax[0].set_ylabel('fraccion acumulada')
ax[0].set_title(f'Casi todo salto < 1 px  (p99 = {np.percentile(jump,99):.0f} px, sin saltos)')

nb = 40; edges = np.linspace(0, fmax, nb + 1); w = edges[1] - edges[0]; ctr = 0.5*(edges[:-1] + edges[1:])
bh, _ = np.histogram(birth, edges); dh, _ = np.histogram(death, edges)
ax[1].bar(ctr - w*0.22, bh, width=w*0.42, color='seagreen', label='nacen')
ax[1].bar(ctr + w*0.22, dh, width=w*0.42, color='indianred', label='mueren')
ax[1].axvspan(f_low, fmax, color='red', alpha=.08)
ax[1].set_xlabel('frame'); ax[1].set_ylabel('tracks por tramo'); ax[1].legend(fontsize=9)
ax[1].set_title('El recambio explota al vaciarse (frente que pasa)')
plt.tight_layout(); plt.show()

### 0.4 Las ráfagas de movimiento del final: ¿reales o artefacto?

Cerca del final aparece una **ráfaga**: de golpe muchas bolitas se mueven a la vez (pico naranja de
0.2). Como el silo ya está casi vacío, uno sospecha que podría ser ruido. Lo ponemos a prueba: una
ráfaga **real** de descarga tiene que verse como caída por gravedad, es decir

1. **coherente hacia abajo** — casi todas con `vy > 0` (en la imagen, y crece hacia abajo),
2. **de tracks continuos** — el movimiento sale de trayectorias sin huecos (`gap == 1`), no de
   reapariciones sueltas,
3. **persistente** — cada bolita se mueve durante muchos frames seguidos, no un parpadeo de 1 frame.

Comparamos la ráfaga contra la descarga **estable** del medio del video. Si se comportan igual, la
ráfaga es una **avalancha real de vaciado**, no un artefacto — sólo que con menos bolitas.

In [ ]:
s['gap'] = g['frame'].diff()
s['mv']  = s['v'] * k > V_MOV
f_burst  = int(mpf.idxmax())                      # frame con mas bolitas moviles = la rafaga mas grande

def caracterizar(a, b, nombre):
    d  = s[(s.frame >= a) & (s.frame <= b)]
    mv = d[d.mv]
    # persistencia: frames consecutivos que cada bolita se mueve dentro de la ventana
    persist = mv.groupby('particle').size()
    return dict(nombre=nombre, n=len(mv),
                abajo=(mv.vy > 0).mean()*100,
                continuo=(mv.gap == 1).mean()*100,
                vy=mv.vy.abs().median()*k,
                persist=persist.median())

R = [caracterizar(2000, 2400, 'descarga estable (medio)'),
     caracterizar(f_burst-15, f_burst+15, f'RAFAGA final (frame {f_burst}, t={f_burst/FPS:.0f}s)')]
print(f'{"ventana":<38}{"%hacia_abajo":>13}{"%continuo":>11}{"med|vy|":>10}{"persist(frames)":>17}')
for r in R:
    print(f'{r["nombre"]:<38}{r["abajo"]:12.0f}%{r["continuo"]:10.0f}%{r["vy"]:8.0f}  {r["persist"]:14.0f}')
print('\n=> la rafaga es hacia abajo, de tracks continuos y persistente, IGUAL que la descarga estable:')
print('   es una AVALANCHA REAL de vaciado, no un artefacto (solo que con menos material).')

# imagen: mapa de vectores de la rafaga (una ventana corta) para verlo
d = s[(s.frame >= f_burst-3) & (s.frame <= f_burst+3) & s.mv]
fig, ax = plt.subplots(figsize=(4.5, 8))
ax.quiver(d.x, d.y, d.vx, d.vy, np.hypot(d.vx, d.vy)*k, cmap='viridis', angles='xy', scale=60, width=.006)
ax.set_ylim(t.y.max()+10, t.y.min()-10); ax.set_aspect('equal')
ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]'); ax.set_title(f'Vectores de la rafaga (t={f_burst/FPS:.0f}s)\ntodos hacia abajo = caida real')
plt.tight_layout(); plt.show()

## 1. Estático vs móvil (borde inferior del rango)

Las bolitas están congeladas ~97% del tiempo. En escala log la distribución es bimodal (estático
+ móvil); el valle da el umbral estático/móvil = **borde inferior** de la transición.

In [ ]:
v = t['v'].values; v = v[v > 0]; logv = np.log10(v)
hist, edges = np.histogram(logv, 120); centers = 0.5*(edges[:-1]+edges[1:])
hs = gaussian_filter1d(hist.astype(float), 3)
static_idx = np.argmax(hs)
mob_region = centers > np.log10(2)
mobile_idx = np.where(mob_region)[0][np.argmax(hs[mob_region])]
v_static = 10 ** centers[static_idx + np.argmin(hs[static_idx:mobile_idx])]   # px/f
print(f'umbral estatico/movil = {v_static:.2f} px/f = {v_static*k:.0f} mm/s   (borde inferior)')

## 2. Limpiar vectores malos — por consistencia temporal

No filtramos por `|vx|>|vy|` (eso borra el movimiento lateral real de la descompresión y no atrapa
los errores). Filtramos por **consistencia temporal**: descartamos un vector si se despega de la
trayectoria de su propia bolita (`dev`) o si viene de un hueco de tracking (`gap>1`). Estándar en
velocimetría de partículas (Westerweel & Scarano 2005).

In [ ]:
s = t.sort_values(['particle', 'frame']).copy()
s['gap'] = s.groupby('particle')['frame'].diff()
s['mv']  = s['v'] > v_static
s['vx_tm'] = s.groupby('particle')['vx'].transform(lambda a: a.rolling(5, center=True, min_periods=3).median())
s['vy_tm'] = s.groupby('particle')['vy'].transform(lambda a: a.rolling(5, center=True, min_periods=3).median())
s['dev'] = np.hypot(s['vx']-s['vx_tm'], s['vy']-s['vy_tm'])
mob_raw = s[s['v'] > v_static]
mob = mob_raw[(mob_raw['dev'] <= 3) & (mob_raw['gap'] == 1)].copy()
print(f'moviles {len(mob_raw)} -> tras limpiar {len(mob)}  ({100*(1-len(mob)/len(mob_raw)):.1f}% descartado)')

## 3. Distribución trimodal → rango de la velocidad de transición

La fase móvil **no** es un solo bloque: tiene un **pico de transición** (bolitas lentas que recién
arrancan) y un **pico de caída** (rápidas). El **valle entre ellos** es el borde superior del rango
de transición.

In [ ]:
mms = mob['v'].values * k
hh, ee = np.histogram(mms, np.linspace(0, 1400, 120)); cc = 0.5*(ee[:-1]+ee[1:])
hh2 = gaussian_filter1d(hh.astype(float), 2)
p_trans = cc[cc < 250][np.argmax(hh2[cc < 250])]                 # pico transicion
fall_m = (cc > 250) & (cc < 900); p_fall = cc[fall_m][np.argmax(hh2[fall_m])]   # pico caida
seg = (cc >= p_trans) & (cc <= p_fall)
v_valley = cc[seg][np.argmin(hh2[seg])]                          # valle = borde superior
lo, hi = v_static*k, v_valley
print(f'pico transicion ~{p_trans:.0f} mm/s | pico caida ~{p_fall:.0f} mm/s | valle = {v_valley:.0f} mm/s')
print(f'==> RANGO DE VELOCIDAD DE TRANSICION = {lo:.0f} - {hi:.0f} mm/s')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(t['v'].values*k, bins=np.linspace(0, 1400, 120), color='steelblue', alpha=.85)
ax.axvline(lo, color='green', ls='--', lw=2, label=f'estatico | transicion = {lo:.0f} mm/s')
ax.axvline(hi, color='red',   ls='--', lw=2, label=f'transicion | caida = {hi:.0f} mm/s')
ax.axvspan(lo, hi, color='orange', alpha=.15)
ax.set_yscale('log'); ax.set_xlabel('v [mm/s]'); ax.set_ylabel('conteo (log)')
ax.set_title('Distribucion trimodal de velocidad'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Chequeo: la transición **es** "empezar a moverse"

Detectamos, por trayectoria, el **onset** (cuando una bolita cruza de estática a móvil). Su
velocidad en ese instante es baja (~borde inferior) y en los frames siguientes **acelera** cruzando
el rango de transición hacia la caída. Confirma que el rango 51–253 mm/s es la fase de arranque.

In [ ]:
s['prev'] = s.groupby('particle')['mv'].shift(1)
s['onset'] = s['mv'] & (s['prev'] == False) & (s['gap'] == 1)
oi = s.index[s['onset']]
print(f'eventos de onset: {len(oi)} | velocidad en el onset = {np.nanmedian(s.loc[oi,"v"])*k:.0f} mm/s')
print('velocidad (mediana) despues del onset, misma bolita:')
for kk in [0, 1, 3, 5, 8, 12]:
    col = s['v'] if kk == 0 else s.groupby('particle')['v'].shift(-kk)
    print(f'  +{kk:2d} frames: {np.nanmedian(col.loc[oi])*k:4.0f} mm/s')

## 5. Mapa de zonas

Coarse-graining en grilla. Clasificamos cada celda por la **velocidad de caída típica (mediana) de
sus bolitas móviles**:
- **Estático**: hay bolitas pero casi no se mueven (móvil < 2% del tiempo).
- **Transición**: velocidad típica dentro del rango 51–253 mm/s (arrancando).
- **Caída**: velocidad típica por encima de 253 mm/s (ya cayendo).

In [ ]:
d = np.load('../outputs/velocity_field.npz'); xb, yb = d['x_bins'], d['y_bins']
nx, ny = len(xb)-1, len(yb)-1
dx, dy = np.diff(xb).mean(), np.diff(yb).mean(); cell_cm2 = (dx/PXMM/10)*(dy/PXMM/10)
ci = lambda col, bins: np.clip(np.digitize(col, bins)-1, 0, len(bins)-2)
t['xi'], t['yi'] = ci(t['x'], xb), ci(t['y'], yb)
mob['xi'], mob['yi'] = ci(mob['x'], xb), ci(mob['y'], yb)
G = lambda ser: ser.unstack(fill_value=0).reindex(index=range(ny), columns=range(nx), fill_value=0).values
ntot = G(t.groupby(['yi', 'xi']).size())
nmob = G(mob.groupby(['yi', 'xi']).size())
Vmed = np.full((ny, nx), np.nan)                       # vel de caida tipica por celda [mm/s]
for (j, i), val in mob.groupby(['yi', 'xi'])['v'].median().items(): Vmed[j, i] = val*k
mobfrac = np.divide(nmob, ntot, out=np.zeros((ny, nx)), where=ntot > 0)

present = ntot > 200
flow    = present & (mobfrac >= 0.02) & (nmob >= 10)
static      = present & (mobfrac < 0.02)
transition  = flow & (Vmed >= lo) & (Vmed < hi)
falling     = flow & (Vmed >= hi)

Z = np.full((ny, nx), np.nan); Z[static] = 0; Z[transition] = 1; Z[falling] = 2
ext = [xb[0], xb[-1], yb[-1], yb[0]]
fig, ax = plt.subplots(1, 2, figsize=(12, 6.5))
ax[0].imshow(Z, cmap=ListedColormap(['#dddddd', 'orange', 'crimson']), origin='upper',
             extent=ext, aspect='equal', interpolation='nearest')
ax[0].set_title('Zonas\ngris=estatico  naranja=TRANSICION  rojo=caida')
im = ax[1].imshow(Vmed, cmap='viridis', origin='upper', extent=ext, aspect='equal')
fig.colorbar(im, ax=ax[1], label='vel tipica [mm/s]'); ax[1].set_title('Velocidad de caida tipica')
for a in ax: a.set_xlabel('x [px]'); a.set_ylabel('y [px]')
plt.tight_layout(); plt.show()

## 6. Área de la zona de transición (descompresión)

In [ ]:
A = lambda m: m.sum() * cell_cm2
print(f'celda = {cell_cm2:.2f} cm^2')
print('=== AREAS (cm^2) ===')
print(f'  estatico   : {A(static):6.1f}')
print(f'  TRANSICION : {A(transition):6.1f}   <-- ZONA DE DESCOMPRESION ({transition.sum()} celdas)')
print(f'  caida      : {A(falling):6.1f}')

print('\nsensibilidad al borde superior del rango de transicion:')
for hi_test in [200, 253, 300, 350]:
    tr = flow & (Vmed >= lo) & (Vmed < hi_test)
    print(f'  transicion 51-{hi_test:3.0f} mm/s : {A(tr):6.1f} cm^2 ({tr.sum()} celdas)')

## 7. Animación: la banda de transición en el tiempo

**Toda la descarga** (~7000 frames, submuestreada 1 de cada 4). **Naranja = bolitas en el rango de
transición (51–253 mm/s)**, gris = estático, rojo = caída. Se ve el **frente de descompresión**: las
naranjas aparecen en el borde superior del material móvil — donde el bloque estático se suelta antes
de caer — y debajo cae el rojo. Se guarda en `outputs/nb02/transition_animation.mp4`.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import Video

v_hi_pxf = hi / k                      # borde superior de transicion (hi=253 mm/s) en px/f
t['reg'] = np.where(t['v'] < v_static, 'static',
             np.where(t['v'] < v_hi_pxf, 'trans', 'fall'))

# TODA la descarga (submuestreada 1 de cada `step` frames para que el mp4 sea manejable)
step = 4
fmax = int(t['frame'].max())
frames_win = list(range(0, fmax + 1, step))
by = dict(tuple(t.groupby('frame')))
print(f'animando {len(frames_win)} frames (1 de cada {step}) de 0 a {fmax} = {fmax/FPS:.0f}s reales')

fig, ax = plt.subplots(figsize=(5, 9))
ax.set_xlim(t.x.min()-10, t.x.max()+10); ax.set_ylim(t.y.max()+10, t.y.min()-10)   # y invertido
ax.set_aspect('equal'); ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
scS = ax.scatter([], [], s=5,  c='#cfcfcf', alpha=.5, label='estatico')
scF = ax.scatter([], [], s=14, c='crimson', alpha=.7, label='caida')
scT = ax.scatter([], [], s=45, c='orange', edgecolors='k', linewidths=.3, label='TRANSICION', zorder=3)
ax.legend(loc='upper right', fontsize=8); ttl = ax.set_title('')
EMP = np.empty((0, 2))
def upd(i):
    fr = frames_win[i]; d = by.get(fr)
    for sc, r in [(scS, 'static'), (scF, 'fall'), (scT, 'trans')]:
        sc.set_offsets(d[d['reg'] == r][['x', 'y']].values if d is not None and (d['reg'] == r).any() else EMP)
    ttl.set_text(f'frame {fr} | t={fr/FPS:.1f}s   (naranja = 51-253 mm/s)')
    return scS, scF, scT, ttl
ani = FuncAnimation(fig, upd, frames=len(frames_win), interval=50, blit=True)
ani.save(f'{OUTDIR}/transition_animation.mp4', writer='ffmpeg', fps=25, dpi=80)
plt.close()
Video(f'{OUTDIR}/transition_animation.mp4', embed=True)

## 8. ¿Qué encontramos? (en criollo)

- La descarga es a los tirones: casi todo **quieto**, y de a ratos un **paquete** de bolitas arranca.
- Cuando una bolita arranca lo hace **despacito** (~63 mm/s) y en pocos frames **acelera** hasta
  caer (~500 mm/s). Esas dos velocidades aparecen como **dos picos separados** en los datos → el
  rango de **transición/descompresión** es **51–253 mm/s**, sacado de los propios datos.
- Espacialmente, la **zona de transición está arriba y en los bordes** del canal: es el frente
  donde el material se suelta del bloque estático antes de caer. Debajo ya es caída.
- El **área de esa zona** es el número que pedía el director (ver §6); depende un poco de dónde se
  pone el borde superior del rango, por eso mostramos la sensibilidad.
- La animación (§7) confirma el comportamiento: las bolitas de transición marcan el frente móvil.
- Nadie había medido esto para hidrogeles en régimen intermitente desde video → contribución nueva.